In [ ]:
%reload_ext autoreload
%autoreload 2
import numpy as np
from instruction_generator import InstructionGenerator

instruction_generator = InstructionGenerator("../../episodes", "../../data/episode_data")


In [ ]:
# result = instruction_generator.generate_instruction("nv_apartment", 8)
result = instruction_generator.generate_instruction("grCommercial_scene11", 10)


In [ ]:
# get episodes that are not bad (multiprocessing)
import os
import json
import multiprocessing as mp

scene_folder = os.path.join(os.path.expanduser("~"), "data", "isaac_scenes_v1")
episode_data_folder = os.path.join(scene_folder, "episode_data")

def process_episode(task):
    scene_name, episode_name = task
    episode_path = os.path.join(episode_data_folder, scene_name, episode_name)
    if not os.path.isdir(episode_path):
        return None
    episode_id = int(episode_name.replace("episode_", ""))
    gen_path = os.path.join(episode_path, "generated_instructions.json")
    if not os.path.exists(gen_path):
        print("=" * 20)
        print(f"no generated instructions: {scene_name} {episode_name}")
        instruction_generator.generate_instruction(scene_name, episode_id)
        return "missing"
    with open(gen_path, "r") as f:
        data = json.load(f)
    if not data.get("checked", False):
        print("=" * 20)
        print(f"not checked: {scene_name} {episode_name}")
        instruction_generator.generate_instruction(scene_name, episode_id)
        return "unchecked"
    return None

tasks = []
for scene_name in os.listdir(episode_data_folder):
    if scene_name.startswith("grCommercial"):
        scene_path = os.path.join(episode_data_folder, scene_name)
        for episode_name in os.listdir(scene_path):
            tasks.append((scene_name, episode_name))

num_workers = max(1, (os.cpu_count() or 1) - 1)
ctx = mp.get_context("fork")
with ctx.Pool(processes=num_workers) as pool:
    for _ in pool.imap_unordered(process_episode, tasks):
        pass


In [ ]:
import os

i = 0
instruction_generator.vlm_based_generation(result[i]["episode"], result[i]["aligned_instructions"])

In [ ]:
# get episodes that are not bad (multiprocessing)
import os
import json
import multiprocessing as mp

scene_folder = os.path.join(os.path.expanduser("~"), "data", "isaac_scenes_v1")
episode_data_folder = os.path.join(scene_folder, "episode_data")

def process_episode(task):
    scene_name, episode_name = task
    episode_path = os.path.join(episode_data_folder, scene_name, episode_name)
    if not os.path.isdir(episode_path):
        return None
    episode_id = int(episode_name.replace("episode_", ""))
    gen_path = os.path.join(episode_path, "generated_instructions.json")
    if not os.path.exists(gen_path):
        print("=" * 20)
        print(f"no generated instructions: {scene_name} {episode_name}")
        instruction_generator.generate_instruction(scene_name, episode_id)
        return "missing"
    with open(gen_path, "r") as f:
        data = json.load(f)
    if not data.get("checked", False):
        print("=" * 20)
        print(f"not checked: {scene_name} {episode_name}")
        instruction_generator.generate_instruction(scene_name, episode_id)
        return "unchecked"
    return None

tasks = []
for scene_name in os.listdir(episode_data_folder):
    if scene_name.startswith("grCommercial"):
        scene_path = os.path.join(episode_data_folder, scene_name)
        for episode_name in os.listdir(scene_path):
            tasks.append((scene_name, episode_name))

num_workers = max(1, (os.cpu_count() or 1) - 1)
ctx = mp.get_context("fork")
with ctx.Pool(processes=num_workers) as pool:
    for _ in pool.imap_unordered(process_episode, tasks):
        pass
